In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV, KFold
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import (
    concordance_index_censored,
    cumulative_dynamic_auc,
)
from sksurv.util import Surv


# ============================================================
# 1. Load data
# ============================================================

DATA_PATH = r'Path/dataML.csv'

TIME_COL = "TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING"
EVENT_COL = "MET_STATUS"

data = pd.read_csv(DATA_PATH)

# Check required columns
required_cols = [TIME_COL, EVENT_COL]
missing_cols = [col for col in required_cols if col not in data.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Ensure survival times and event status are numeric
data[TIME_COL] = pd.to_numeric(data[TIME_COL], errors="raise")
data[EVENT_COL] = pd.to_numeric(data[EVENT_COL], errors="raise")

if data[TIME_COL].isna().any() or data[EVENT_COL].isna().any():
    raise ValueError("Missing survival times or event statuses.")

if (data[TIME_COL] < 0).any():
    raise ValueError("Survival times cannot be negative.")

if not set(data[EVENT_COL].unique()).issubset({0, 1}):
    raise ValueError("MET_STATUS must be coded 0 (censored) or 1 (event).")

# Features must be numeric and contain no missing values.
X = data.drop(columns=[TIME_COL, EVENT_COL]).copy()

non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise ValueError(
        f"Encode categorical predictors before training: {non_numeric}"
    )

if X.isna().any().any():
    raise ValueError(
        "Predictors contain missing values. Impute them within the "
        "cross-validation pipeline before fitting the models."
    )

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Predictors contain infinite values.")

if X.shape[1] == 0:
    raise ValueError("No predictor columns were found.")

y = Surv.from_dataframe(EVENT_COL, TIME_COL, data)

print(f"Number of patients: {len(data)}")
print(f"Number of predictors: {X.shape[1]}")
print(f"Number of metastasis events: {int(data[EVENT_COL].sum())}")


# ============================================================
# 2. Hyperparameter grid
# ============================================================

param_grid = {
    "n_estimators": [100, 500],
    "max_depth": [10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [3, 5],
    "max_features": ["sqrt", None],
    "bootstrap": [True],
}


# ============================================================
# 3. Nested cross-validation configuration
# ============================================================

N_SPLITS = 5
RANDOM_STATE = 42

outer_cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Materialize folds so the same folds are used throughout.
outer_splits = list(outer_cv.split(X))

nested_c_indices = []
best_params_per_fold = []
fold_auc_results = []

# Store fold-specific results for reproducibility.
fold_results = []


# ============================================================
# 4. Determine a common time-dependent AUC evaluation window
# ============================================================

# Find a common follow-up interval supported by all outer
# training and test folds. Shrink both ends slightly to avoid
# evaluating exactly at the boundaries.

lower_bounds = []
upper_bounds = []

for train_idx, test_idx in outer_splits:

    train_times = y[TIME_COL][train_idx]
    test_times = y[TIME_COL][test_idx]

    lower_bounds.append(
        max(train_times.min(), test_times.min())
    )
    upper_bounds.append(
        min(train_times.max(), test_times.max())
    )

common_min = max(lower_bounds)
common_max = min(upper_bounds)

if not common_max > common_min:
    raise ValueError(
        "No common follow-up interval exists across all folds. "
        "Consider fewer folds or a more appropriate evaluation design."
    )

# Avoid exact boundaries, where censoring estimates can be unstable.
margin = 0.05 * (common_max - common_min)

safe_min = common_min + margin
safe_max = common_max - margin

if safe_max <= safe_min:
    safe_min = common_min
    safe_max = common_max

# Ten common evaluation times.
eval_times = np.linspace(safe_min, safe_max, 10)

print("\nTime-dependent AUC evaluation times:")
print(np.round(eval_times, 3))


# ============================================================
# 5. Nested cross-validation
# ============================================================

for fold_idx, (train_idx, test_idx) in enumerate(outer_splits):

    print("\n" + "=" * 60)
    print(f"OUTER FOLD {fold_idx + 1}/{N_SPLITS}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # --------------------------------------------------------
    # Inner CV: hyperparameter tuning on outer training data
    # --------------------------------------------------------

    inner_cv = KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rsf = RandomSurvivalForest(
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    grid_search = GridSearchCV(
        estimator=rsf,
        param_grid=param_grid,
        cv=inner_cv,
        scoring=None,  # RSF.score() uses the concordance index
        n_jobs=-1,
        refit=True,
        error_score="raise",
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print("\nBest parameters:")
    print(best_params)

    best_params_per_fold.append(best_params)

    # --------------------------------------------------------
    # Outer test fold: C-index
    # --------------------------------------------------------

    risk_scores = best_model.predict(X_test)

    c_index = concordance_index_censored(
        y_test[EVENT_COL],
        y_test[TIME_COL],
        risk_scores,
    )[0]

    print(f"\nOuter test C-index: {c_index:.4f}")

    nested_c_indices.append(float(c_index))

    # --------------------------------------------------------
    # Outer test fold: time-dependent AUC
    # --------------------------------------------------------

    # Higher risk = greater predicted probability of an event.
    predicted_survival_functions = (
        best_model.predict_survival_function(X_test)
    )

    # Shape: (n_test_patients, n_evaluation_times)
    auc_risk_scores = np.asarray([
        [1.0 - survival_fn(t) for t in eval_times]
        for survival_fn in predicted_survival_functions
    ])

    # Estimate censoring distribution using outer training data.
    # Evaluate discrimination using only the held-out test fold.
    try:
        auc_values, integrated_auc = cumulative_dynamic_auc(
            y_train,
            y_test,
            auc_risk_scores,
            eval_times,
        )

    except ValueError as exc:
        raise ValueError(
            f"AUC calculation failed in outer fold {fold_idx + 1}. "
            "Check follow-up support, censoring distribution, and "
            "event counts at the evaluation times. Original error: "
            f"{exc}"
        ) from exc

    fold_auc_results.append(
        [float(value) for value in auc_values]
    )

    print("\nTime-dependent AUC:")
    for t, auc in zip(eval_times, auc_values):
        print(f"  Time {t:.3f}: AUC = {auc:.4f}")

    print(f"Integrated AUC for this fold: {integrated_auc:.4f}")

    fold_results.append({
        "outer_fold": fold_idx + 1,
        "c_index": float(c_index),
        "auc_by_time": [float(v) for v in auc_values],
        "integrated_auc": float(integrated_auc),
        "best_params": best_params,
    })


# ============================================================
# 6. Summarize nested cross-validation results
# ============================================================

mean_nested_cindex = float(np.mean(nested_c_indices))
std_nested_cindex = float(np.std(nested_c_indices, ddof=1))

fold_auc_results = np.asarray(fold_auc_results, dtype=float)

mean_auc_by_time = np.mean(fold_auc_results, axis=0)
std_auc_by_time = np.std(fold_auc_results, axis=0, ddof=1)

print("\n" + "=" * 60)
print("NESTED CROSS-VALIDATION RESULTS")
print("=" * 60)

print("Outer-fold C-indices:", np.round(nested_c_indices, 4))
print(f"Mean C-index: {mean_nested_cindex:.4f}")
print(f"SD C-index:   {std_nested_cindex:.4f}")

print("\nTime-dependent AUC across outer folds:")
for t, mean_auc, std_auc in zip(
    eval_times,
    mean_auc_by_time,
    std_auc_by_time,
):
    print(
        f"Time {t:.3f}: "
        f"AUC = {mean_auc:.4f} ± {std_auc:.4f}"
    )


# ============================================================
# 7. Plot time-dependent AUC
# ============================================================

plt.figure(figsize=(8, 6))

plt.errorbar(
    eval_times,
    mean_auc_by_time,
    yerr=std_auc_by_time,
    marker="o",
    capsize=4,
    label="Nested CV mean ± SD",
)

plt.axhline(
    0.5,
    linestyle="--",
    label="Chance discrimination",
)

plt.xlabel("Time")
plt.ylabel("Time-dependent AUC")
plt.title("RSF: Time-dependent AUC across outer CV folds")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(
    "rsf_nested_cv_time_dependent_auc.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 8. Final hyperparameter search on the full dataset
# ============================================================

print("\nRunning final grid search on the full dataset...")

final_cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

final_grid_search = GridSearchCV(
    estimator=RandomSurvivalForest(
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    param_grid=param_grid,
    cv=final_cv,
    scoring=None,
    n_jobs=-1,
    refit=True,
    error_score="raise",
)

final_grid_search.fit(X, y)

final_best_params = final_grid_search.best_params_

print("\nBest parameters from full-dataset CV:")
print(final_best_params)


# ============================================================
# 9. Train and save the final model
# ============================================================

final_model = RandomSurvivalForest(
    **final_best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_model.fit(X, y)

joblib.dump(final_model, "final_rsf_model.pkl")

print("\nFinal RSF model trained on the entire dataset.")
print("Saved model: final_rsf_model.pkl")


# ============================================================
# 10. Save all results
# ============================================================

results = {
    "nested_cv_fold_c_indices": nested_c_indices,
    "nested_cv_mean_c_index": mean_nested_cindex,
    "nested_cv_std_c_index": std_nested_cindex,
    "best_params_per_outer_fold": best_params_per_fold,
    "outer_fold_results": fold_results,
    "auc_evaluation_times": [float(t) for t in eval_times],
    "auc_by_fold": fold_auc_results,
    "mean_auc_by_time": [
        float(v) for v in mean_auc_by_time
    ],
    "std_auc_by_time": [
        float(v) for v in std_auc_by_time
    ],
    "final_best_params_full_dataset": final_best_params,
}

with open("nested_cv_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved results: nested_cv_results.json")
print("Saved AUC plot: rsf_nested_cv_time_dependent_auc.png")
